<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/ses/notebooks/08_financial_mart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import re

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"

DATASETS_DIR = "datasets"
# RECONCILIAÇÃO: Mudando a entrada para herdar estritamente os Best-Sellers reais do Código 7
INPUT_FILENAME = f"7_{CONSOLE_NAME}_games_bestsellers.xlsx"   # Puxa apenas os sucessos comerciais
OUTPUT_FILENAME = f"{CONSOLE_NAME}_financial_performance.xlsx"  # O artefato de BI limpo (Sem número)

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def clean_human_strings(text):
    """Splits strings with corporate notes/multiple companies and returns only the primary name."""
    if not text or pd.isna(text):
        return "Unknown"

    text = str(text)
    # Remove anotações regionais e publishers secundários ex: "Capcom (JP/NA) Nintendo (PAL)" -> "Capcom"
    parts = re.split(r'\(|/|•', text)
    clean_name = parts[0].strip()

    # Arranca possíveis resíduos de colchetes de notas da Wikipédia
    clean_name = re.sub(r'\[.*\]', '', clean_name).strip()
    return clean_name

def pipeline_bi_visualization(input_path, output_path):
    """
    Cleans up redundant features and builds a lightweight, human-readable
    dataset optimized for charts, dashboards, and visual analysis.
    """
    print(f"💎 [CODE 08] Manufacturing Visual Analysis Artifact: {CONSOLE_NAME.upper()}")

    # 1. Pipeline gatekeeper check
    if not os.path.exists(input_path):
        print(f"❌ Critical Error: Input file '{input_path}' not found. Please run Notebook 07 first.")
        return None

    # 2. Load the best-sellers data (Zero-biased records are already excluded)
    df = pd.read_excel(input_path)
    print(f"📊 Integrating {len(df)} real commercial hits into the visualization template...")

    # 3. Apply Humanized Cleaning to Publishers and Developers
    df['Publisher'] = df['Publisher'].apply(clean_human_strings)
    df['Developer'] = df['Developer'].apply(clean_human_strings)

    # 4. Feature Selection and Rename (Enforcing your clean analytical schema)
    df_bi = pd.DataFrame({
        'Title': df['T1_Clean'],
        'Publisher': df['Publisher'],
        'Developer': df['Developer'],
        'Category': df['Category_Primary'].fillna('Unclassified'),
        'Release': df['Original_Release'],
        'Score': df['IGDB_Score'],
        'Sold': df['Copies_Sold_Millions']
    })

    # 5. Save the final unnumbered BI spreadsheet
    df_bi.to_excel(output_path, index=False)

    print("\n" + "═"*60)
    print(f"🚀 VISUAL TEMPLATE GENERATED SUCCESSFULLY!")
    print(f"📊 Target shape: {df_bi.shape[0]} rows x {df_bi.shape[1]} columns.")
    print(f"💾 File saved for direct BI/Dashboard ingestion at: {output_path}")
    print("═"*60 + "\n")

    # Preview to verify the aesthetic of the final columns
    print("👀 PREVIEW OF THE BI OPTIMIZED DATASET:")
    return df_bi.head(5)

# Trigger the optimization
df_visual_analysis_preview = pipeline_bi_visualization(INPUT_PATH, OUTPUT_PATH)